# Farneback

Mac: `small-movts` Micromamba env

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from src.utils_farneback import calc_intensities_farneback, save_txt

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

# "demo.mp4" for speed
video_fname = "28-MAY-2025_00h04m33.969s.mp4"
data_dir = Path("data/farneback/subj1_cam1")
video_path = data_dir / video_fname

In [ ]:
# 10m 38s on Mac for 5m-long video
# 12s for 6s-long video "demo.mp4"
times, intensities = calc_intensities_farneback(video_path)

In [ ]:
plt.plot(times, intensities)
plt.xlabel("Time (s)")
plt.ylabel("Motion intensity (pixels/s)")
plt.tight_layout()
plt.savefig(video_path.parent / f"{video_path.stem}_farneback_intensity.pdf")

In [ ]:
times_path = data_dir / f"{video_path.stem}_times.txt"
intensities_path = data_dir / f"{video_path.stem}_intensities.txt"

In [ ]:
# save_txt(times, times_path)
# save_txt(intensities, intensities_path)

In [ ]:
from typing import Final
from src.utils_spectral_power import smoothen
import numpy as np
import matplotlib.pyplot as plt

times_np = np.loadtxt(times_path) # times in seconds, 1D array
intensities_np = np.loadtxt(intensities_path)

# calc sliding average
periods = (# 1, 2, 5,
           8, 12
           )  # in seconds
time_delta: Final[float] = times_np[1] - times_np[0]

for p in periods:
    n_points, averages = smoothen(intensities_np, period=p, time_delta=time_delta)
    plt.figure()
    plt.plot(times_np[n_points:], averages)

# EEG analysis

Mac: `femba` Micromamba env

In [ ]:
import mne
from src.video_utils import parse_timestamp_from_filename

time_str_video, sec_start_video = parse_timestamp_from_filename(video_fname)
time_str_video

In [ ]:
eeg_fname = "27-MAY-2025_22h19m28.399s.edf"
time_str_eeg, sec_start_eeg = parse_timestamp_from_filename(eeg_fname)
time_str_eeg

In [ ]:
video_offset_sec = sec_start_video - sec_start_eeg
video_offset_sec

In [ ]:
from pathlib import Path

eeg_path = Path("data/farneback") / eeg_fname
raw = mne.io.read_raw_edf(str(eeg_path))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.utils_spectral_power import alpha_delta_powers, alpha_band, delta_band

duration_sec = 5 * 60
times, alpha_power, delta_power, raw_fragment = alpha_delta_powers(raw, duration_sec, video_offset_sec + duration_sec / 2)

In [ ]:
raw_fragment.plot()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(times, alpha_power)
axes[0].set_ylabel("Alpha power ($V^2$)")
axes[0].set_title(f"Alpha-band power ({alpha_band[0]}–{alpha_band[1]} Hz)")
axes[0].set_ylim([0, 1e-10])
axes[0].grid(True, alpha=0.3)

axes[1].plot(times, delta_power)
axes[1].set_ylabel("Delta power ($V^2$)")
axes[1].set_xlabel("Time from start of video-aligned fragment (s)")
axes[1].set_title(f"Delta-band power ({delta_band[0]}–{delta_band[1]} Hz)")
axes[1].set_ylim([0, 0.4*1e-9])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(video_path.parent / f"{video_path.stem}_alpha_delta_powers.pdf")